In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlightDelayAnalysis") \
    .getOrCreate()

parquet_path = "../data/sample/parquet/"

df = spark.read.parquet(parquet_path)

df.printSchema()

df.show(5, truncate=False)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/10/21 14:04:14 WARN Utils: Your hostname, codespaces-90a6fa, resolves to a loopback address: 127.0.0.1; using 10.0.11.10 instead (on interface eth0)
25/10/21 14:04:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/21 14:04:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/21 14:04:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


root
 |-- DestAirportID: integer (nullable = true)
 |-- OriginAirportID: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FlightDate: date (nullable = true)
 |-- Marketing_Airline_Network: string (nullable = true)
 |-- DOT_ID_Marketing_Airline: integer (nullable = true)
 |-- Operating_Airline : string (nullable = true)
 |-- DOT_ID_Operating_Airline: integer (nullable = true)
 |-- Flight_Number_Operating_Airline: integer (nullable = true)
 |-- OriginAirportSeqID: integer (nullable = true)
 |-- OriginCityMarketID: integer (nullable = true)
 |-- Origin: string (nullable = true)
 |-- OriginCityName: string (nullable = true)
 |-- DestAirportSeqID: integer (nullable = true)
 |-- DestCityMarketID: integer (nullable = true)
 |-- Dest: string (nullable = true)
 |-- DestCityName: string (nullable = true)
 |-- CRSDepTime: integer (nullable = true)
 |-- DepT

25/10/21 14:04:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------+---------------+----+-----+----------+---------+----------+-------------------------+------------------------+------------------+------------------------+-------------------------------+------------------+------------------+------+--------------+----------------+----------------+----+------------+----------+-------+--------+---------------+--------+--------------------+----------+-------+---------+--------+------+----------+-------+--------+---------------+--------+------------------+----------+---------+----------------+--------+------------+------------+--------+-------------+-----------------+----------+----------------+--------+----------------+-------------------+-------------------+-------------------+-------------------+----------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+-------------------+---------------+---------------+

In [2]:
from pyspark.sql.functions import avg, count, col

origin_weather_delay_stats = df.filter(col("WeatherDelay") > 0) \
    .groupBy(
        "OriginWindDirection", 
        "OriginWindSpeed", 
        "OriginWindGusts", 
        "OriginVisibility", 
        "OriginPrecipitation", 
        "OriginClouds", 
        "OriginTemperature", 
        "OriginDewPoint"
    ) \
    .agg(
        avg("WeatherDelay").alias("AvgWeatherDelayMinutes"),
        count("*").alias("NumWeatherDelays")
    ) \
    .orderBy(col("AvgWeatherDelayMinutes").desc())

origin_weather_delay_stats.show(truncate=False)

+-------------------+---------------+---------------+----------------+-------------------+-------------------------+-----------------+--------------+----------------------+----------------+
|OriginWindDirection|OriginWindSpeed|OriginWindGusts|OriginVisibility|OriginPrecipitation|OriginClouds             |OriginTemperature|OriginDewPoint|AvgWeatherDelayMinutes|NumWeatherDelays|
+-------------------+---------------+---------------+----------------+-------------------+-------------------------+-----------------+--------------+----------------------+----------------+
|350                |6              |NULL           |NULL            |[]                 |[few clouds]             |NULL             |NULL          |436.0                 |1               |
|NULL               |NULL           |NULL           |NULL            |[]                 |[]                       |NULL             |NULL          |249.0                 |1               |
|0                  |0              |NULL         

In [3]:
dest_weather_delay_stats = df.filter(col("WeatherDelay") > 0) \
    .groupBy(
        "DestWindDirection",
        "DestWindSpeed",
        "DestWindGusts",
        "DestVisibility",
        "DestPrecipitation",
        "DestClouds",
        "DestTemperature",
        "DestDewPoint"
    ) \
    .agg(
        avg("WeatherDelay").alias("AvgWeatherDelayMinutes"),
        count("*").alias("NumWeatherDelays")
    ) \
    .orderBy(col("AvgWeatherDelayMinutes").desc())

dest_weather_delay_stats.show(truncate=False)

+-----------------+-------------+-------------+--------------+-----------------+------------------------------------------------+---------------+------------+----------------------+----------------+
|DestWindDirection|DestWindSpeed|DestWindGusts|DestVisibility|DestPrecipitation|DestClouds                                      |DestTemperature|DestDewPoint|AvgWeatherDelayMinutes|NumWeatherDelays|
+-----------------+-------------+-------------+--------------+-----------------+------------------------------------------------+---------------+------------+----------------------+----------------+
|340              |3            |NULL         |NULL          |[]               |[broken clouds, overcast]                       |NULL           |NULL        |436.0                 |1               |
|100              |7            |NULL         |NULL          |[]               |[few clouds, overcast]                          |NULL           |NULL        |332.0                 |1               |
|250 

In [4]:
from pyspark.sql.functions import when

df_with_wind_bucket = df.withColumn(
    "OriginWindSpeedBucket",
    when(col("OriginWindSpeed") < 5, "<5 mph")
    .when((col("OriginWindSpeed") >= 5) & (col("OriginWindSpeed") < 15), "5-15 mph")
    .when((col("OriginWindSpeed") >= 15) & (col("OriginWindSpeed") < 25), "15-25 mph")
    .otherwise("25+ mph")
)

wind_speed_delay_stats = df_with_wind_bucket.filter(col("WeatherDelay") > 0) \
    .groupBy("OriginWindSpeedBucket") \
    .agg(
        avg("WeatherDelay").alias("AvgWeatherDelayMinutes"),
        count("*").alias("NumWeatherDelays")
    ) \
    .orderBy("OriginWindSpeedBucket")

wind_speed_delay_stats.show(truncate=False)

+---------------------+----------------------+----------------+
|OriginWindSpeedBucket|AvgWeatherDelayMinutes|NumWeatherDelays|
+---------------------+----------------------+----------------+
|25+ mph              |249.0                 |1               |
|5-15 mph             |119.83333333333333    |6               |
|<5 mph               |168.66666666666666    |3               |
+---------------------+----------------------+----------------+



In [ ]:
from pyspark.sql.functions import when, col, sum

visibility_buckets = df.withColumn(
    "OriginVisibilityBucket",
    when(col("OriginVisibility").isNull(), "Unknown")
    .when(col("OriginVisibility") < 1, "<1 mile")
    .when(col("OriginVisibility") < 3, "1-3 miles")
    .when(col("OriginVisibility") < 5, "3-5 miles")
    .otherwise(">5 miles")
)

visibility_delay = visibility_buckets.groupBy("OriginVisibilityBucket").agg(
    count("*").alias("NumFlights"),
    round(avg("WeatherDelay"), 2).alias("AvgWeatherDelay"),
    round((sum(when(col("WeatherDelay") > 0, 1).otherwise(0)) / count("*")) * 100, 2).alias("WeatherDelayPercent")
).orderBy("AvgWeatherDelay", ascending=False)

visibility_delay.show()

+----------------------+----------+---------------+-------------------+
|OriginVisibilityBucket|NumFlights|AvgWeatherDelay|WeatherDelayPercent|
+----------------------+----------+---------------+-------------------+
|               Unknown|      1882|           4.03|               0.53|
+----------------------+----------+---------------+-------------------+



In [8]:
from pyspark.sql.functions import explode, col, avg, count

delayed_flights = df.filter(col("WeatherDelay") > 0)

cloud_delay_data = delayed_flights.select(
    "DestClouds", "WeatherDelay", "DestCityName"
).withColumn("CloudType", explode("DestClouds"))

cloud_impact = cloud_delay_data.groupBy("CloudType").agg(
    count("*").alias("NumWeatherDelays"),
    avg("WeatherDelay").alias("AvgWeatherDelayMinutes")
).orderBy(col("AvgWeatherDelayMinutes").desc())

cloud_impact.show(truncate=False)

+----------------+----------------+----------------------+
|CloudType       |NumWeatherDelays|AvgWeatherDelayMinutes|
+----------------+----------------+----------------------+
|overcast        |3               |339.0                 |
|broken clouds   |4               |223.25                |
|scattered clouds|4               |132.75                |
|few clouds      |5               |92.6                  |
|clear           |3               |75.66666666666667     |
+----------------+----------------+----------------------+



In [ ]:

from pyspark.sql.functions import explode, col, when, avg, count, sum as _sum, round

exploded_clouds = df.withColumn("OriginCloudType", explode("OriginClouds"))

clouds_with_label = exploded_clouds.withColumn(
    "IsDelayed",
    when(col("WeatherDelay") > 0, 1).otherwise(0)
)

cloud_delay_summary = clouds_with_label.groupBy("OriginCloudType").agg(
    count("*").alias("TotalFlights"),
    _sum("IsDelayed").alias("DelayedFlights"),
    round((_sum("IsDelayed") / count("*")) * 100, 2).alias("DelayPercent"),
    round(avg("WeatherDelay"), 2).alias("AvgWeatherDelayMins")
).orderBy("DelayPercent", ascending=False)

cloud_delay_summary.show(truncate=False)

+----------------+------------+--------------+------------+-------------------+
|OriginCloudType |TotalFlights|DelayedFlights|DelayPercent|AvgWeatherDelayMins|
+----------------+------------+--------------+------------+-------------------+
|overcast        |385         |4             |1.04        |2.32               |
|broken clouds   |591         |2             |0.34        |1.19               |
|clear           |452         |1             |0.22        |1.24               |
|few clouds      |868         |1             |0.12        |2.93               |
|scattered clouds|305         |0             |0.0         |0.0                |
+----------------+------------+--------------+------------+-------------------+



In [18]:
wind_binned = df.withColumn(
    "OriginWindSpeedBucket",
    when(col("OriginWindSpeed").isNull(), "Unknown")
    .when(col("OriginWindSpeed") < 5, "<5 kt")
    .when(col("OriginWindSpeed") < 10, "5-10 kt")
    .when(col("OriginWindSpeed") < 15, "10-15 kt")
    .otherwise("15+ kt")
)

wind_delay = wind_binned.groupBy("OriginWindSpeedBucket").agg(
    count("*").alias("NumFlights"),
    round(avg("WeatherDelay"), 2).alias("AvgWeatherDelay"),
    round((sum(when(col("WeatherDelay") > 0, 1).otherwise(0)) / count("*")) * 100, 2).alias("WeatherDelayPercent")
).orderBy("OriginWindSpeedBucket", ascending=False)

wind_delay.show()

+---------------------+----------+---------------+-------------------+
|OriginWindSpeedBucket|NumFlights|AvgWeatherDelay|WeatherDelayPercent|
+---------------------+----------+---------------+-------------------+
|              Unknown|       108|          15.56|               0.93|
|                <5 kt|       751|           3.67|                0.4|
|              5-10 kt|       804|           3.84|               0.62|
|               15+ kt|        14|            0.0|                0.0|
|             10-15 kt|       205|           1.93|               0.49|
+---------------------+----------+---------------+-------------------+



In [ ]:
from pyspark.sql.functions import when, count, avg, sum as _sum, round

dest_wind_bucketed = df.withColumn(
    "DestWindSpeedBucket",
    when(col("DestWindSpeed").isNull(), "Unknown")
    .when(col("DestWindSpeed") < 5, "<5 kt")
    .when(col("DestWindSpeed") < 10, "5-10 kt")
    .when(col("DestWindSpeed") < 15, "10-15 kt")
    .otherwise("15+ kt")
)

dest_wind_delay = dest_wind_bucketed.groupBy("DestWindSpeedBucket").agg(
    count("*").alias("NumFlights"),
    round(avg("WeatherDelay"), 2).alias("AvgWeatherDelay"),
    round((_sum(when(col("WeatherDelay") > 0, 1).otherwise(0)) / count("*")) * 100, 2).alias("WeatherDelayPercent")
).orderBy("DestWindSpeedBucket", ascending=False)

dest_wind_delay.show()

+-------------------+----------+---------------+-------------------+
|DestWindSpeedBucket|NumFlights|AvgWeatherDelay|WeatherDelayPercent|
+-------------------+----------+---------------+-------------------+
|            Unknown|        61|            2.8|               3.28|
|              <5 kt|       348|          11.81|               0.57|
|            5-10 kt|       717|           3.92|               0.42|
|             15+ kt|       253|           1.51|                0.4|
|           10-15 kt|       503|           1.45|                0.4|
+-------------------+----------+---------------+-------------------+



In [27]:
df_with_delay_flag = df.withColumn(
    "IsDelayed", when(col("ArrDel15") == 1, 1).otherwise(0)
)

dest_wind_delay_summary = df_with_delay_flag.groupBy("DestCityName").agg(
    count("*").alias("TotalFlights"),
    sum("IsDelayed").alias("DelayedFlights"),
    round(avg("DestWindSpeed"), 2).alias("AvgDestWindSpeed"),
    round((sum("IsDelayed") / count("*")) * 100, 2).alias("DelayRatePercent")
).orderBy("AvgDestWindSpeed", ascending=False)

dest_wind_delay_summary.show(truncate=False)

+---------------------+------------+--------------+----------------+----------------+
|DestCityName         |TotalFlights|DelayedFlights|AvgDestWindSpeed|DelayRatePercent|
+---------------------+------------+--------------+----------------+----------------+
|New York, NY         |266         |66            |12.68           |24.81           |
|Dallas/Fort Worth, TX|365         |82            |9.61            |22.47           |
|Chicago, IL          |373         |55            |9.53            |14.75           |
|Denver, CO           |246         |49            |8.65            |19.92           |
|Atlanta, GA          |418         |72            |7.76            |17.22           |
|Los Angeles, CA      |214         |42            |4.17            |19.63           |
+---------------------+------------+--------------+----------------+----------------+



In [26]:
from pyspark.sql.functions import col, size, explode, array_join, when, count, sum, round, avg

df_with_delay_flag = df.withColumn(
    "IsDelayed", when(col("ArrDel15") == 1, 1).otherwise(0)
)

df_precip = df_with_delay_flag.withColumn(
    "PrecipitationLabel",
    when(size(col("OriginPrecipitation")) == 0, "None")
    .otherwise(array_join(col("OriginPrecipitation"), ", "))
)

precip_delay_summary = df_precip.groupBy("PrecipitationLabel").agg(
    count("*").alias("TotalFlights"),
    sum("IsDelayed").alias("DelayedFlights"),
    round((sum("IsDelayed") / count("*")) * 100, 2).alias("DelayRatePercent"),
    round(avg("ArrDelay"), 2).alias("AvgArrDelayMinutes")
).orderBy("DelayRatePercent", ascending=False)

precip_delay_summary.show(truncate=False)

+------------------+------------+--------------+----------------+------------------+
|PrecipitationLabel|TotalFlights|DelayedFlights|DelayRatePercent|AvgArrDelayMinutes|
+------------------+------------+--------------+----------------+------------------+
|rain, mist        |2           |2             |100.0           |142.0             |
|mist              |105         |38            |36.19           |20.19             |
|haze              |4           |1             |25.0            |-7.75             |
|None              |1769        |325           |18.37           |5.22              |
|NULL              |2           |0             |0.0             |-3.5              |
+------------------+------------+--------------+----------------+------------------+



In [ ]:
from pyspark.sql.functions import col, size, when, count, sum, round

df_with_flags = df.withColumn(
    "HasPrecipitation", when(size(col("DestPrecipitation")) > 0, 1).otherwise(0)
).withColumn(
    "IsDelayed", when(col("ArrDel15") == 1, 1).otherwise(0)
)

destination_weather_delay_stats = df_with_flags.groupBy("DestCityName").agg(
    count("*").alias("TotalFlights"),
    sum("HasPrecipitation").alias("FlightsWithPrecip"),
    round((sum("HasPrecipitation") / count("*")) * 100, 2).alias("PrecipFrequencyPercent"),
    sum("IsDelayed").alias("DelayedFlights"),
    round((sum("IsDelayed") / count("*")) * 100, 2).alias("DelayRatePercent")
).orderBy(col("PrecipFrequencyPercent").desc())

destination_weather_delay_stats.show(truncate=False)

+---------------------+------------+-----------------+----------------------+--------------+----------------+
|DestCityName         |TotalFlights|FlightsWithPrecip|PrecipFrequencyPercent|DelayedFlights|DelayRatePercent|
+---------------------+------------+-----------------+----------------------+--------------+----------------+
|Los Angeles, CA      |214         |19               |8.88                  |42            |19.63           |
|Dallas/Fort Worth, TX|365         |21               |5.75                  |82            |22.47           |
|Chicago, IL          |373         |21               |5.63                  |55            |14.75           |
|Atlanta, GA          |418         |23               |5.5                   |72            |17.22           |
|Denver, CO           |246         |10               |4.07                  |49            |19.92           |
|New York, NY         |266         |3                |1.13                  |66            |24.81           |
+---------